# Sorting Algorithms: Time and Space Comparison
Analysis of Algorithms | Assignment 3

This notebook implements Selection Sort, Bubble Sort, Quick Sort and Merge Sort from scratch, and tests each one on 4 arrays: size 5 and size 100, both sorted and reverse sorted (16 runs in total). For every run it records the time (average of 3 runs), the number of comparisons and the extra memory used.

## 1. The four algorithms

In [1]:
import time, sys, tracemalloc, random
import pandas as pd

def selection_sort(arr):
    """Selection Sort: find the smallest item in the unsorted part and put it in front."""
    n = len(arr)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:      # comparison
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]   # swap
    # Extra space: O(1) -> in-place


def bubble_sort(arr):
    """Bubble Sort: compare neighbours and swap them if they are in the wrong order.
    No early-exit flag, so it always does all the passes."""
    n = len(arr)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:        # comparison
                arr[j], arr[j + 1] = arr[j + 1], arr[j]   # swap
    # Extra space: O(1) -> in-place


def quick_sort(arr, low=0, high=None):
    """Quick Sort: pick the last item as pivot, put smaller items on its left and
    bigger items on its right, then sort both sides the same way."""
    if high is None:
        high = len(arr) - 1
    if low < high:
        pivot = arr[high]
        i = low - 1
        for j in range(low, high):
            if arr[j] <= pivot:            # comparison
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i + 1], arr[high] = arr[high], arr[i + 1]
        p = i + 1                          # final position of the pivot
        quick_sort(arr, low, p - 1)
        quick_sort(arr, p + 1, high)
    # Extra space: O(log n) stack on average, O(n) stack in the worst case


def merge_sort(arr):
    """Merge Sort: split in two halves, sort each half, then merge them."""
    if len(arr) <= 1:
        return
    mid = len(arr) // 2
    left = arr[:mid]                       # extra arrays are created here
    right = arr[mid:]
    merge_sort(left)
    merge_sort(right)
    i = j = k = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:            # comparison
            arr[k] = left[i]; i += 1
        else:
            arr[k] = right[j]; j += 1
        k += 1
    while i < len(left):
        arr[k] = left[i]; i += 1; k += 1
    while j < len(right):
        arr[k] = right[j]; j += 1; k += 1
    # Extra space: O(n) -> needs extra arrays


algorithms = {
    "Selection Sort": selection_sort,
    "Bubble Sort": bubble_sort,
    "Quick Sort": quick_sort,
    "Merge Sort": merge_sort,
}

# quick correctness test against Python's sorted()
for name, f in algorithms.items():
    for size in [0, 1, 2, 5, 10, 100]:
        for _ in range(20):
            a = [random.randint(0, 1000) for _ in range(size)]
            b = a.copy(); f(b)
            assert b == sorted(a), f"{name} failed"
    print(name, ": OK")

Selection Sort : OK
Bubble Sort : OK
Quick Sort : OK
Merge Sort : OK


## 2. The 16 runs
Each timed run sorts 500 fresh copies of the array and divides by 500, so that very small times can be measured. This is repeated 3 times and the average is taken. Copying the array is not part of the timing.

In [2]:
cases = {
    "Case 1: size 5, sorted":           list(range(1, 6)),
    "Case 2: size 5, reverse sorted":   list(range(5, 0, -1)),
    "Case 3: size 100, sorted":         list(range(1, 101)),
    "Case 4: size 100, reverse sorted": list(range(100, 0, -1)),
}
RUNS = 3        # assignment: run 3 times and take the average
REPS = 500      # each run sorts 500 fresh copies and divides, so tiny times are measurable

class Item:
    """Wrapper that counts every comparison made by a sorting algorithm."""
    count = 0
    def __init__(self, v): self.v = v
    def __lt__(self, o): Item.count += 1; return self.v < o.v
    def __gt__(self, o): Item.count += 1; return self.v > o.v
    def __le__(self, o): Item.count += 1; return self.v <= o.v
    def __ge__(self, o): Item.count += 1; return self.v >= o.v

def time_one_run(func, data):
    copies = [data.copy() for _ in range(REPS)]     # copying is NOT timed
    start = time.perf_counter_ns()
    for c in copies:
        func(c)
    end = time.perf_counter_ns()
    return (end - start) / REPS / 1000              # microseconds per sort

def count_comparisons(func, data):
    Item.count = 0
    func([Item(x) for x in data])
    return Item.count

def extra_memory(func, data):
    arr = data.copy()
    tracemalloc.start()
    func(arr)
    peak = tracemalloc.get_traced_memory()[1]       # peak extra memory in bytes
    tracemalloc.stop()
    return peak

def max_depth(func, data):
    """Deepest recursion level reached (stack space used by recursive sorts)."""
    state = {"d": 0, "max": 0}
    name = func.__name__
    def prof(frame, event, arg):
        if frame.f_code.co_name == name:
            if event == "call":
                state["d"] += 1; state["max"] = max(state["max"], state["d"])
            elif event == "return":
                state["d"] -= 1
    sys.setprofile(prof); func(data.copy()); sys.setprofile(None)
    return state["max"]

for f in algorithms.values():          # warm-up
    f(list(range(100, 0, -1)))

tables = {}
for case, data in cases.items():
    rows = []
    for name, func in algorithms.items():
        t = [time_one_run(func, data) for _ in range(RUNS)]
        depth = max_depth(func, data) if name in ("Quick Sort", "Merge Sort") else 1
        rows.append({
            "Algorithm": name,
            "Run 1 (µs)": round(t[0], 2), "Run 2 (µs)": round(t[1], 2), "Run 3 (µs)": round(t[2], 2),
            "Average (µs)": round(sum(t) / RUNS, 2),
            "Comparisons": count_comparisons(func, data),
            "Extra memory (bytes)": extra_memory(func, data),
            "Recursion depth": depth,
        })
    tables[case] = pd.DataFrame(rows).set_index("Algorithm")
    print(case); display(tables[case]); print()

# save the tables as CSV files for GitHub
for i, (case, df) in enumerate(tables.items(), start=1):
    df.to_csv(f"table_case{i}.csv")

Case 1: size 5, sorted


,Run 1 (µs),Run 2 (µs),Run 3 (µs),Average (µs),Comparisons,Extra memory (bytes),Recursion depth
Algorithm,,,,,,,
Selection Sort,1.13,1.01,1.02,1.06,10,128,1
Bubble Sort,0.95,1.07,0.94,0.99,10,128,1
Quick Sort,2.11,2.07,2.16,2.11,10,88,5
Merge Sort,2.88,2.73,2.63,2.75,5,80,4



Case 2: size 5, reverse sorted


,Run 1 (µs),Run 2 (µs),Run 3 (µs),Average (µs),Comparisons,Extra memory (bytes),Recursion depth
Algorithm,,,,,,,
Selection Sort,2.07,1.48,1.13,1.56,10,128,1
Bubble Sort,1.59,1.66,1.55,1.60,10,128,1
Quick Sort,2.52,1.98,1.87,2.12,10,88,5
Merge Sort,2.78,2.98,2.80,2.85,7,80,4



Case 3: size 100, sorted


,Run 1 (µs),Run 2 (µs),Run 3 (µs),Average (µs),Comparisons,Extra memory (bytes),Recursion depth
Algorithm,,,,,,,
Selection Sort,200.86,198.45,194.74,198.02,4950,128,1
Bubble Sort,238.86,233.66,229.15,233.89,4950,128,1
Quick Sort,419.46,387.72,393.41,400.20,4950,88,100
Merge Sort,115.32,120.24,115.83,117.13,316,1608,8



Case 4: size 100, reverse sorted


,Run 1 (µs),Run 2 (µs),Run 3 (µs),Average (µs),Comparisons,Extra memory (bytes),Recursion depth
Algorithm,,,,,,,
Selection Sort,219.96,205.98,206.32,210.75,4950,128,1
Bubble Sort,487.26,461.29,482.30,476.95,4950,128,1
Quick Sort,769.74,488.13,289.27,515.71,4950,88,100
Merge Sort,106.45,103.68,110.17,106.77,356,1608,8
